## RAGの構築

In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [3]:
import sys
!{sys.executable} -m pip install langchain-community

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 22.6 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 18.2 MB/s eta 0:00:00
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

# PDFファイルを読込
loader = DirectoryLoader('./data/pdf', glob="./*.pdf",   loader_cls=PyPDFLoader)
documents = loader.load()

# 結果の表示
print(documents)

[Document(metadata={'producer': 'Skia/PDF m80', 'creator': 'Chromium', 'creationdate': '2024-11-12T08:21:15+00:00', 'moddate': '2024-11-12T08:21:15+00:00', 'source': 'data\\pdf\\01就業規則.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='みらいテクノロジー株式会社  就業規則\nみらいテクノロジー株式会社では、従業員の皆さんが安⼼して働けるよう、働き⽅に関する基本\n的なルールを設けています。この就業規則は、勤務時間や休暇の取り⽅、遅刻や⽋勤などに関す\nるルールを明確にし、皆さんがより働きやすい環境を提供するために定められています。\n1. 勤務時間\n1. 通常の勤務時間\n勤務時間は、⽉曜⽇から⾦曜⽇まで、午前 9 時から午後 6 時までの 8 時間です。\n1 ⽇のうち、 1 時間は休憩時間として確保しています。\n2. 休憩時間\n休憩は、正午 12 時から午後 1 時までの 1 時間です。この時間帯には、⾷事やリフレッ\nシュのために⾃由に過ごしてください。\n業務の都合で休憩時間をずらす場合があるので、その際は上司に確認してくださ\nい。\n3. 勤務形態の種類\n当社には通常勤務のほか、リモートワークも可能です。リモートワークの場合も、\n基本的に勤務時間や休憩時間はオフィス勤務と同じです。\n出勤や退勤の時間が固定されない「フレックスタイム制度」も⼀部導⼊しています\nが、フレックスタイムで働く⽅は、始業と終業の時間帯を事前に上司と相談し、許\n可を得てください。\n2. 出勤・退勤\n1. 出勤・退勤の記録\n出勤時と退勤時には、必ず会社のタイムカードシステムを使って記録してくださ\nい。\nリモートワーク時には、オンラインシステム上で打刻する必要があります。\n2. 遅刻・早退\n出勤時間に遅れる場合や、予定より早く退勤する場合は、必ず事前に上司へ連絡を\nしてください。\n遅刻や早退が多発する場合は、上司との⾯談が必要になることがあります。\n3. ⽋勤（病⽋やその他の理由によ

In [3]:
from langchain_text_splitters import CharacterTextSplitter
import tiktoken

# 言語モデルに合うトークナイザー名を取得
encoding_name = tiktoken.encoding_for_model(MODEL_NAME).name

# テキスト分割を作成
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(encoding_name)

# チャンクに分割
texts = text_splitter.split_documents(documents)

# チャンク数と内容の表示
print("texts_size=", len(texts))
for txt in texts[:3]:
    print(txt)
    print("-" * 50)

texts_size= 17
page_content='みらいテクノロジー株式会社  就業規則
みらいテクノロジー株式会社では、従業員の皆さんが安⼼して働けるよう、働き⽅に関する基本
的なルールを設けています。この就業規則は、勤務時間や休暇の取り⽅、遅刻や⽋勤などに関す
るルールを明確にし、皆さんがより働きやすい環境を提供するために定められています。
1. 勤務時間
1. 通常の勤務時間
勤務時間は、⽉曜⽇から⾦曜⽇まで、午前 9 時から午後 6 時までの 8 時間です。
1 ⽇のうち、 1 時間は休憩時間として確保しています。
2. 休憩時間
休憩は、正午 12 時から午後 1 時までの 1 時間です。この時間帯には、⾷事やリフレッ
シュのために⾃由に過ごしてください。
業務の都合で休憩時間をずらす場合があるので、その際は上司に確認してくださ
い。
3. 勤務形態の種類
当社には通常勤務のほか、リモートワークも可能です。リモートワークの場合も、
基本的に勤務時間や休憩時間はオフィス勤務と同じです。
出勤や退勤の時間が固定されない「フレックスタイム制度」も⼀部導⼊しています
が、フレックスタイムで働く⽅は、始業と終業の時間帯を事前に上司と相談し、許
可を得てください。
2. 出勤・退勤
1. 出勤・退勤の記録
出勤時と退勤時には、必ず会社のタイムカードシステムを使って記録してくださ
い。
リモートワーク時には、オンラインシステム上で打刻する必要があります。
2. 遅刻・早退
出勤時間に遅れる場合や、予定より早く退勤する場合は、必ず事前に上司へ連絡を
してください。
遅刻や早退が多発する場合は、上司との⾯談が必要になることがあります。
3. ⽋勤（病⽋やその他の理由による休み）
病気ややむを得ない理由で⽋勤する場合は、できる限り早めに上司や⼈事部へ連絡
してください。
無断⽋勤は、会社の業務に⽀障をきたすため、必ず避けてください。無断⽋勤が続
く場合、懲戒処分の対象となる場合があります。
3. 残業・休⽇出勤' metadata={'producer': 'Skia/PDF m80', 'creator': 'Chromium', 'creationdate': '2024-11-12T08:21:15+00:00', 'moddate': '2024-11-12T

In [6]:
import sys
!{sys.executable} -m pip install langchain-chroma

  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached filelock-3.24.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached oauthlib-3.3.1-p

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] T

In [4]:
# インデックスの構築
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# エンベディングモデルの指定
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# インデックスの構築
db = Chroma.from_documents(texts, embedding_model)

In [5]:
# Retrieverの作成
retriever = db.as_retriever()

# 検索の実施
results = retriever.invoke("有給休暇の付与日数は？")

# 結果を表示
for result in results:
    print(result.page_content)
    print("-" * 50)

みらいテクノロジー株式会社  休暇規則
みらいテクノロジー株式会社では、従業員の皆さんが仕事と⽣活のバランスを保ちながら働ける
よう、さまざまな休暇制度を設けています。この休暇規則は、休暇の種類や取得⽅法、条件など
を明確にし、安⼼して休暇を利⽤していただくためのものです。
1. 年次有給休暇（有給休暇）
1. 有給休暇とは
有給休暇は、給与を受け取りながら休暇を取得できる制度です。
⼼⾝のリフレッシュや私⽤のために⾃由に利⽤できます。
2. 付与⽇数
⼊社から 6 ヶ⽉継続勤務し、全労働⽇の 8 割以上出勤した場合に、初めて有給休暇が
付与されます。
初年度は 10 ⽇間の有給休暇が付与され、その後は勤続年数に応じて増加します。
勤続年数 年次有給休暇⽇数
0.5 年 10 ⽇
1.5 年 11 ⽇
2.5 年 12 ⽇
3.5 年 14 ⽇
4.5 年 16 ⽇
5.5 年 18 ⽇
6.5 年以上 20 ⽇
3. 有給休暇の取得⽅法
有給休暇を取得する際は、原則として3 ⽇前までに上司に申請してください。
緊急の場合は、当⽇の申請も可能ですが、できるだけ早めに連絡をお願いします。
申請は、社内の休暇申請システムを利⽤してください。
4. 有給休暇の繰越し
未使⽤の有給休暇は、翌年度に限り繰り越すことができます。
最⼤で 40 ⽇間の有給休暇を保有することが可能です。
2. 特別休暇
特別休暇は、有給休暇とは別に特定の事情に応じて取得できる休暇です。
1. 慶弔休暇
結婚休暇︓本⼈が結婚する場合、5 ⽇間の休暇が取得できます。
--------------------------------------------------
配偶者の出産休暇︓配偶者が出産する場合、2 ⽇間の休暇が取得できます。
忌引休暇︓家族が亡くなった場合、親等に応じて以下の休暇が取得できます。
配偶者、⼦、親︓5 ⽇間
兄弟姉妹、祖⽗⺟︓3 ⽇間
配偶者の親︓3 ⽇間
2. 産前産後休暇
産前休暇︓出産予定⽇の6 週間前から取得可能です。
産後休暇︓出産⽇の翌⽇から8 週間は就業が禁⽌されています。
産前産後休暇中は、健康保険から出産⼿当⾦が⽀給されます。
3. 育児休業
⼦供が1 歳になるまでの間、育児休業を取得できます。
保育所に⼊れないなどの事情がある場合、最⻑で2 歳まで延⻑可能

In [6]:
# 保存先を指定
db = Chroma.from_documents(texts, embedding_model, persist_directory="./chroma_db")

In [7]:
# ストレージから復元
db = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# プロンプトテンプレートの作成
prompt = ChatPromptTemplate.from_template("""提供されたコンテキストのみに基づいて次の質問に答えてください:

<コンテキスト>
{context}
</コンテキスト>

Question: {input}""")

In [9]:
# モデルの作成
chat_model = ChatOpenAI(model_name=MODEL_NAME)

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = ({"context": retriever, "input": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser())

In [11]:
# チェーンの実行
response = chain.invoke("有給休暇の付与日数は？")

# 結果を表示
print(response)

有給休暇の付与日数は以下の通りです：

- 勤続0.5年：10日
- 勤続1.5年：11日
- 勤続2.5年：12日
- 勤続3.5年：14日
- 勤続4.5年：16日
- 勤続5.5年：18日
- 勤続6.5年以上：20日

なお、有給休暇は入社から6ヶ月継続勤務し、全労働日の8割以上出勤した場合に初めて付与されます。
